# v7 — Fine-tune ต่อจาก v6 ด้วยข้อมูลเพิ่ม (ภาพแปลงใหม่ 5–6 แปลง) · โปรเจ็กต์ weed-detect-l2bhf · เทรนเป็นรอบ ๆ รอบละ ≤ 5 ชม.

**ตอบคำถาม: ข้อมูลแปลงใหม่ช่วยเท่าไหร่ และโมเดลใช้กับแปลงที่ไม่เคยเห็นได้แค่ไหน**
เริ่มจาก `v6_best_sofar.pt` · lr0 1e-4 · 40 epochs · patience 15 · ชิ้นภาพ 1024 และค่าอื่นเหมือน v6 · เทียบ v6 กับ v7 บน val/test ชุดเดียวกัน
แยกผลเป็น **แปลงเดิม** กับ **แปลงใหม่** (อ่าน tag `test_new_field` / `valid_new_field` จาก Roboflow)

**เตรียม (Kaggle):** Add Input **weed-v5-files** (ชุด Weed-only) และ **train_v6_relabel** (มี v6_best_sofar.pt) · Secret `ROBOFLOW_API_KEY` · GPU T4 · Internet On

**รัน:** cell 1–3 เช็กตัวเลข → **Save & Run All (Commit)** → ถ้าขึ้น "⏸ ยังไม่ครบ" → Add Input = Output ของ version ล่าสุดของโน้ตบุ๊กนี้ → Save & Run All อีกครั้ง → จนขึ้น "✅ เทรนครบแล้ว" → โหลด `v7_results.zip`
**กติกา:** v7 ต้องดีกว่า v6 บน val (SAHI 1024) อย่างน้อย +1.0 ไม่งั้นใช้ v6 · วัด test ครั้งเดียว

In [ ]:
# 1) ติดตั้ง + หาไฟล์
!pip install ultralytics sahi 2>&1 | tail -3
import os, glob, shutil, yaml, math, random, time, numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
assert 'KAGGLE_KERNEL_RUN_TYPE' in os.environ, 'โน้ตบุ๊กนี้ทำไว้สำหรับ Kaggle'
assert torch.cuda.is_available(), 'ไม่มี GPU — ตั้ง Accelerator เป็น GPU T4'
print('GPU:', torch.cuda.get_device_name(0))

T1 = time.time()
K = '/kaggle/input'
# หาไฟล์แบบเร็ว: ลอง path ที่รู้อยู่แล้วก่อน → ไม่เจอค่อยค้นแบบจำกัดความลึก (ค้นทั้ง /kaggle/input แบบ ** ช้ามาก เพราะมีภาพเป็นหมื่นไฟล์)
KNOWN = {
    'v6':   [f'{K}/notebooks/attawitmadsait/train-v6-relabel/v6_best_sofar.pt'],
    'data': [f'{K}/datasets/attawitmadsait/weed-v5-files/weed_thai_v3_dataset', f'{K}/weed-v5-files/weed_thai_v3_dataset'],
}
def find(patterns, maxdepth=6):
    for p in patterns:
        for d in range(1, maxdepth + 1):
            hit = sorted(glob.glob(f'{K}/' + '*/' * (d - 1) + p))
            if hit: return hit[0]
    return None
def first_existing(paths): return next((p for p in paths if os.path.exists(p)), None)

V6_PT = first_existing(KNOWN['v6']) or find(['v6_best_sofar.pt'])              # โมเดล v6 (จุดเริ่มต้น + ตัวเทียบ)
OLD = first_existing(KNOWN['data']) or (os.path.dirname(os.path.dirname(find(['weed_thai_v3_dataset/train/images/rf_*']) or '')) or None)
print('v6  :', V6_PT); print('OLD :', OLD); print(f'หาไฟล์ใช้เวลา {time.time()-T1:.0f} วินาที')
assert V6_PT and OLD, 'หาไฟล์ไม่เจอ — ตรวจ Add Input (train_v6_relabel, weed-v5-files)'
USE_ROBOFLOW = True              # False = label เดิม v3 (weed-v5-files) · True = ดึงเวอร์ชันล่าสุดจาก Roboflow
ROBOFLOW_VERSION = 2             # ← version สำหรับเทรนของ weed-detect-l2bhf (train 1005 / valid 166 / test 140)
RF_PROJECT = 'weed-detect-l2bhf'
PREFIX_TEST_NEW, PREFIX_VALID_NEW = [], []   # สำรองเมื่ออ่าน tag ไม่ได้ เช่น ['DJI_20260915'] (ส่วนต้นชื่อไฟล์ของแปลง test/valid ใหม่)
CHUNK_H = 5.0                    # เทรนรอบละไม่เกิน 5 ชม. แล้วเก็บไฟล์ resume (Kaggle ตัดที่ 12 ชม. · งานที่ถูกตัด/Stop จะไม่มี Output)
RESUME_IN = find(['v7_resume_last.pt'])     # มีไฟล์นี้ใน Input = เทรนต่อจากรอบก่อน
PREV_BEST = find(['v7_best_sofar.pt'])
PREV_CSV  = find(['v7_results_progress.csv'])
print('เทรนต่อจาก:', RESUME_IN or '— เริ่มใหม่ (รอบแรก)')

TILE, OVERLAP, MIN_VIS, BG_KEEP, IMGSZ = 1024, 0.2, 0.3, 0.3, 1024           # ค่าเดียวกับ v5
TILED = '/tmp/weed_thai_tiled'                                               # ไม่เก็บลง output (ไฟล์ใหญ่)
W = '/kaggle/working'; RUN_NAME = 'weed_thai_v7'

In [ ]:
# 2) ข้อมูล: label เดิม v3 (ค่าเริ่มต้น) หรือดึงเวอร์ชันล่าสุดจาก Roboflow (USE_ROBOFLOW=True)
if not USE_ROBOFLOW:
    OUT, RF_VER = OLD, 'v3 (weed-v5-files)'
    print('ใช้ label เดิม v3:', {sp: len(glob.glob(f'{OUT}/{sp}/images/rf_*')) for sp in ['train', 'valid', 'test']}, '(ควรเป็น 509 / 128 / 111)')
else:
    import subprocess; subprocess.run(['pip', '-q', 'install', 'roboflow'])
    from kaggle_secrets import UserSecretsClient
    from roboflow import Roboflow
    KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY').strip().strip('"')
    proj = Roboflow(api_key=KEY).workspace('attawit-madsait').project(RF_PROJECT)
    vers = sorted(int(str(v.version).split('/')[-1]) for v in proj.versions())
    RF_VER = ROBOFLOW_VERSION or vers[-1]
    print('เวอร์ชันที่มี:', vers, '→ ใช้ version', RF_VER)
    RF_DIR = f'/tmp/rf_v{RF_VER}'
    if not os.path.exists(RF_DIR):
        proj.version(RF_VER).download('yolov8', location=RF_DIR)

    # รายชื่อภาพแปลงใหม่ใน valid/test จาก tag (ใช้แยกผลแปลงเดิม/แปลงใหม่)
    norm = lambda n: os.path.basename(n).replace('rf_', '', 1).split('.rf.')[0].rsplit('.', 1)[0].lower().removesuffix('_jpg').removesuffix('_jpeg').removesuffix('_png')
    NEW_FIELD = {}
    for tg in ['test_new_field', 'valid_new_field']:
        names = set()
        try:
            for page in proj.search_all(tag=tg, fields=['id', 'name']):
                for it in (page if isinstance(page, list) else [page]):
                    if isinstance(it, dict) and it.get('name'): names.add(norm(it['name']))
        except Exception as e:
            print('⚠️ อ่าน tag', tg, 'ไม่ได้:', str(e)[:150])
        NEW_FIELD[tg] = names; print(f'tag {tg}: {len(names)} ภาพ')
    # สำรอง: ถ้าอ่าน tag ไม่ได้ ใช้ส่วนต้นของชื่อไฟล์ (วันเวลา) ที่ใส่ไว้ใน PREFIX_* (cell 1)
    for tg, pref in [('test_new_field', PREFIX_TEST_NEW), ('valid_new_field', PREFIX_VALID_NEW)]:
        if not NEW_FIELD[tg] and pref:
            NEW_FIELD[tg] = {norm(f) for sp in ['valid', 'test'] for f in glob.glob(f'{RF_DIR}/{sp}/images/*')
                             if any(os.path.basename(f).lower().startswith(x.lower()) for x in pref)}
            print(f'ใช้ชื่อไฟล์แทน tag {tg}: {len(NEW_FIELD[tg])} ภาพ')

    def to_box_line(l):                       # polygon (x y x y ...) → กรอบ · ทุก class รวมเป็น 0 (weed)
        v = l.split()
        if len(v) == 5: return '0 ' + ' '.join(v[1:])
        if len(v) >= 7 and len(v) % 2 == 1:
            xs, ys = list(map(float, v[1::2])), list(map(float, v[2::2]))
            x1, x2, y1, y2 = max(min(xs), 0), min(max(xs), 1), max(min(ys), 0), min(max(ys), 1)
            if x2 > x1 and y2 > y1: return f'0 {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {x2-x1:.6f} {y2-y1:.6f}'
        return None

    OUT = '/tmp/weed_thai_new'; shutil.rmtree(OUT, ignore_errors=True)
    cnt = {}
    for sp in ['train', 'valid', 'test']:
        os.makedirs(f'{OUT}/{sp}/images'); os.makedirs(f'{OUT}/{sp}/labels')
        for img in glob.glob(f'{OLD}/{sp}/images/w_*'):                          # weed-only ชุดเดิม (ไม่เปลี่ยน)
            b = os.path.basename(img); st = b.rsplit('.', 1)[0]
            os.symlink(img, f'{OUT}/{sp}/images/{b}')
            lb = f'{OLD}/{sp}/labels/{st}.txt'
            if os.path.exists(lb): os.symlink(lb, f'{OUT}/{sp}/labels/{st}.txt')
        n_rf = 0
        for img in glob.glob(f'{RF_DIR}/{sp}/images/*'):                         # นาไทย label ใหม่
            b = 'rf_' + os.path.basename(img); st = b.rsplit('.', 1)[0]
            os.symlink(img, f'{OUT}/{sp}/images/{b}')
            lb = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
            lines = [x for x in (to_box_line(l) for l in (open(lb).read().splitlines() if os.path.exists(lb) else [])) if x]
            open(f'{OUT}/{sp}/labels/{st}.txt', 'w').write('\n'.join(lines)); n_rf += 1
        cnt[sp] = dict(นาไทย_ใหม่=n_rf, weed_only=len(glob.glob(f'{OLD}/{sp}/images/w_*')))
    print(pd.DataFrame(cnt))          # ควรเป็น นาไทย 1005 / 166 / 140

In [ ]:
# 3) ตัดภาพนาไทยเป็นชิ้น — โค้ดเดียวกับ v5 ทุกบรรทัด (seed 0)
random.seed(0)
def positions(L, T):
    if L <= T: return [0]
    n = math.ceil((L - T) / (T * (1 - OVERLAP))) + 1
    return [round(i * (L - T) / (n - 1)) for i in range(n)]

def read_boxes(lbl, W_, H_):
    out = []
    if os.path.exists(lbl):
        for l in open(lbl).read().splitlines():
            v = l.split()
            if len(v) == 5:
                _, cx, cy, w, h = map(float, v)
                out.append(((cx-w/2)*W_, (cy-h/2)*H_, (cx+w/2)*W_, (cy+h/2)*H_))
    return out

shutil.rmtree(TILED, ignore_errors=True)
for split in ['train', 'valid']:
    di, dl = f'{TILED}/{split}/images', f'{TILED}/{split}/labels'
    os.makedirs(di, exist_ok=True); os.makedirs(dl, exist_ok=True)
    n_whole = n_tiles = n_bg_skip = 0
    for img in sorted(glob.glob(f'{OUT}/{split}/images/*')):
        base = os.path.basename(img); stem = os.path.splitext(base)[0]
        lbl = f'{OUT}/{split}/labels/{stem}.txt'
        im = Image.open(img); W_, H_ = im.size
        if not base.startswith('rf_') or max(W_, H_) <= TILE * 1.25:
            os.symlink(img, f'{di}/{base}')
            if os.path.exists(lbl): os.symlink(lbl, f'{dl}/{stem}.txt')
            n_whole += 1; continue
        im = im.convert('RGB'); boxes = read_boxes(lbl, W_, H_)
        for y in positions(H_, TILE):
            for x in positions(W_, TILE):
                tw, th = min(TILE, W_), min(TILE, H_)
                lines = []
                for x1, y1, x2, y2 in boxes:
                    a1, b1, a2, b2 = max(x1, x), max(y1, y), min(x2, x+tw), min(y2, y+th)
                    if a2 <= a1 or b2 <= b1: continue
                    if (a2-a1)*(b2-b1) < MIN_VIS * (x2-x1)*(y2-y1): continue
                    lines.append(f'0 {((a1+a2)/2-x)/tw:.6f} {((b1+b2)/2-y)/th:.6f} {(a2-a1)/tw:.6f} {(b2-b1)/th:.6f}')
                if not lines and random.random() > BG_KEEP:
                    n_bg_skip += 1; continue
                name = f'{stem}_t{x}_{y}'
                im.crop((x, y, x+tw, y+th)).save(f'{di}/{name}.jpg', quality=95)
                open(f'{dl}/{name}.txt', 'w').write('\n'.join(lines)); n_tiles += 1
    print(split, dict(ภาพเต็ม=n_whole, ชิ้น=n_tiles, ชิ้นว่างที่ข้าม=n_bg_skip))   # v5: train 1,149 + 2,940 · valid 120 + 757
yaml.safe_dump({'path': TILED, 'train': 'train/images', 'val': 'valid/images', 'nc': 1, 'names': ['weed']},
               open(f'{TILED}/data.yaml', 'w'))

In [ ]:
# 4) v7 = Fine-tune ต่อจาก v6 ด้วยข้อมูลเพิ่ม เป็นช่วง ๆ (รอบละ ≤ CHUNK_H ชม.) — lr0 0.001 · 100 epochs · patience 30 · ค่าอื่นเหมือน v5
from ultralytics import YOLO
T0 = time.time(); TIME_STOP = False
RESUME_OUT, BEST_OUT = f'{W}/v7_resume_last.pt', f'{W}/v7_best_sofar.pt'
if PREV_BEST: shutil.copy(PREV_BEST, BEST_OUT)
if PREV_CSV:  shutil.copy(PREV_CSV, f'{W}/v7_results_previous_rounds.csv')

def on_epoch_end(tr):
    global TIME_STOP
    el = (time.time() - T0) / 3600; ep = tr.epoch + 1
    if os.path.exists(tr.last): shutil.copy(tr.last, RESUME_OUT)          # สำเนาที่ resume ได้ (ก่อนถูกตัด optimizer ตอนจบ)
    if os.path.exists(tr.best): shutil.copy(tr.best, BEST_OUT)            # best.pt จะถูกเขียนใหม่เฉพาะตอนดีขึ้น
    if os.path.exists(tr.csv):  shutil.copy(tr.csv, f'{W}/v7_results_progress.csv')
    print(f'epoch {ep}/{tr.epochs} · รอบนี้ {el:.2f} ชม. · {el*60/max(ep - START_EP, 1):.1f} นาที/epoch', flush=True)
    if el > CHUNK_H and ep < tr.epochs:
        TIME_STOP = True; tr.stop = True
        print(f'⏸ ครบ {CHUNK_H} ชม. → หยุดรอบนี้ที่ epoch {ep} (เทรนต่อได้)', flush=True)

ARGS = dict(data=f'{TILED}/data.yaml', epochs=40, imgsz=IMGSZ, batch=8, optimizer='AdamW', lr0=0.0001, lrf=0.01,
            warmup_epochs=1, patience=15, mosaic=1.0, flipud=0.5, degrees=15, seed=0, deterministic=True,
            plots=True, project=W, name=RUN_NAME, exist_ok=True, device=0, workers=4)
if RESUME_IN:                              # เทรนต่อ: แก้ path ใน checkpoint ให้ตรงกับเครื่องรอบนี้
    ck = torch.load(RESUME_IN, map_location='cpu', weights_only=False)
    START_EP = int(ck.get('epoch', -1)) + 1
    ck['train_args'].update(data=f'{TILED}/data.yaml', project=W, name=RUN_NAME, save_dir=f'{W}/{RUN_NAME}', device=0, workers=4)
    torch.save(ck, f'{W}/resume_in.pt'); del ck
    print('เทรนต่อจาก epoch', START_EP + 1, flush=True)
    m = YOLO(f'{W}/resume_in.pt'); m.add_callback('on_fit_epoch_end', on_epoch_end); m.train(resume=True)
else:
    START_EP = 0
    m = YOLO(V6_PT); m.add_callback('on_fit_epoch_end', on_epoch_end); m.train(**ARGS)     # เริ่มจากน้ำหนัก v6

TRAIN_DONE = not TIME_STOP
V7_PT = BEST_OUT
if TRAIN_DONE:
    print('✅ เทรนครบแล้ว → cell ถัดไปจะวัดผล | best:', V7_PT)
else:
    print('⏸ ยังไม่ครบ → Save Version เสร็จแล้ว: สร้างรอบใหม่ แล้ว Add Input = Output ของ version นี้ (มี v7_resume_last.pt) แล้ว Save & Run All อีกครั้ง')

In [ ]:
# 5) ตัววัดผล (เหมือนโน้ตบุ๊ก v5 และกลุ่ม C ทุกอย่าง: conf 0.01, จับคู่แบบ ultralytics, ap_per_class)
if not TRAIN_DONE:
    print('ข้าม — ยังเทรนไม่ครบ')
else:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    from ultralytics.utils.metrics import ap_per_class, box_iou
    IOUV = np.linspace(0.5, 0.95, 10)

    def gt_boxes(p, W_, H_):
        lb = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        return np.array([[x1, y1, x2, y2] for x1, y1, x2, y2 in read_boxes(lb, W_, H_)], np.float32).reshape(-1, 4)

    def match(gt, pb):
        correct = np.zeros((len(pb), 10), bool)
        if len(gt) == 0 or len(pb) == 0: return correct
        iou = box_iou(torch.tensor(gt), torch.tensor(pb, dtype=torch.float32)).numpy()
        for i, t in enumerate(IOUV):
            gi, pi = np.nonzero(iou >= t)
            if len(gi):
                m = np.stack([gi, pi, iou[gi, pi]], 1)
                m = m[m[:, 2].argsort()[::-1]]
                m = m[np.unique(m[:, 1], return_index=True)[1]]
                m = m[np.unique(m[:, 0], return_index=True)[1]]
                correct[m[:, 1].astype(int), i] = True
        return correct

    def predict(pt, imgs, sahi_slice):
        yolo = YOLO(pt); det = None
        if sahi_slice:
            for mt in ('ultralytics', 'yolov8'):
                try:
                    det = AutoDetectionModel.from_pretrained(model_type=mt, model_path=pt, confidence_threshold=0.01, device='cuda:0', image_size=IMGSZ); break
                except Exception as e: err = e
            if det is None: raise err
        out, ts = {}, []
        for p in imgs:
            im = np.asarray(ImageOps.exif_transpose(Image.open(p)).convert('RGB')); t = time.time()
            if sahi_slice:
                r = get_sliced_prediction(im, det, slice_height=sahi_slice, slice_width=sahi_slice,
                                          overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0)
                d = [[q.bbox.minx, q.bbox.miny, q.bbox.maxx, q.bbox.maxy, q.score.value] for q in r.object_prediction_list]
            else:
                r = yolo.predict(Image.fromarray(im), conf=0.01, imgsz=IMGSZ, device=0, verbose=False)[0]
                d = np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()]
            ts.append(time.time() - t)
            out[p] = (np.asarray(d, np.float32).reshape(-1, 5), im.shape[1], im.shape[0])
        return out, float(np.mean(ts))

    def score(pred):
        TP, CF, TC = [], [], []
        for p, (d, W_, H_) in pred.items():
            g = gt_boxes(p, W_, H_)
            TP.append(match(g, d[:, :4])); CF.append(d[:, 4]); TC.append(np.zeros(len(g)))
        tp, cf, tc = np.concatenate(TP), np.concatenate(CF), np.concatenate(TC)
        o = ap_per_class(tp, cf, np.zeros(len(cf)), tc)
        return dict(mAP50=round(o[5][:, 0].mean()*100, 1), mAP50_95=round(o[5].mean()*100, 1),
                    P=round(o[2].mean()*100, 1), R=round(o[3].mean()*100, 1))

    def split_new_old(paths, tag):
        new = [p for p in paths if norm(p) in NEW_FIELD.get(tag, set())]
        return [p for p in paths if p not in set(new)], new
    va_all = sorted(glob.glob(f'{OUT}/valid/images/rf_*')); te_all = sorted(glob.glob(f'{OUT}/test/images/rf_*'))
    va_old, va_new = split_new_old(va_all, 'valid_new_field'); te_old, te_new = split_new_old(te_all, 'test_new_field')
    if not te_new: print('⚠️ ไม่พบภาพ test แปลงใหม่จาก tag — ตรวจว่าใส่ tag test_new_field ใน Roboflow แล้ว (ผลรวมยังใช้ได้)')
    SETS = {'val นาไทย': va_all,                                   # ใช้เลือก (รวมแปลงเดิม + แปลงใหม่)
            'val แปลงเดิม': va_old, 'val แปลงใหม่': va_new,
            'test รวม': te_all, 'test แปลงเดิม': te_old, 'test แปลงใหม่': te_new,
            'test เดิม (weed-only)': sorted(glob.glob(f'{OLD}/test/images/w_*'))}
    SETS = {k: v for k, v in SETS.items() if v}
    print({k: len(v) for k, v in SETS.items()})

In [ ]:
# 6) เลือกด้วย VAL (ยังไม่ดู test)
if not TRAIN_DONE:
    print('ข้าม — ยังเทรนไม่ครบ')
else:
    CONFIGS = {'v6 · SAHI 1024': (V6_PT, 1024), 'v7 · SAHI 1024': (V7_PT, 1024),
               'v6 · ไม่ใช้ SAHI': (V6_PT, None), 'v7 · ไม่ใช้ SAHI': (V7_PT, None)}
    PRED, SEC, rows = {}, {}, []
    for k, (pt, s) in CONFIGS.items():
        PRED[(k, 'val นาไทย')], SEC[k] = predict(pt, SETS['val นาไทย'], s)
        rows.append({'config': k, **score(PRED[(k, 'val นาไทย')])}); print('✅', k, flush=True)
    VSEL = pd.DataFrame(rows); display(VSEL)
    # val แยกแปลงเดิม/แปลงใหม่ (ข้อมูลประกอบ — ใช้ val รวมในการเลือก)
    for g in [g for g in ['val แปลงเดิม', 'val แปลงใหม่'] if g in SETS]:
        print(g, {k: score({p: PRED[(k, 'val นาไทย')][p] for p in SETS[g]})['mAP50'] for k in CONFIGS if 'SAHI 1024' in k})
    v = dict(zip(VSEL['config'], VSEL['mAP50']))
    gain = v['v7 · SAHI 1024'] - v['v6 · SAHI 1024']
    WINNER = 'v7 · SAHI 1024' if gain >= 1.0 else 'v6 · SAHI 1024'
    print(f'v7 − v6 บน val (SAHI 1024) = {gain:+.1f} จุด → 👉 WINNER: {WINNER}')
    VSEL.to_csv(f'{W}/v7_val.csv', index=False)

In [ ]:
# 7) วัด TEST ครั้งเดียว — ทุก config · แยกแปลงเดิม / แปลงใหม่ / รวม + เวลา/ภาพ
if not TRAIN_DONE:
    print('ข้าม — ยังเทรนไม่ครบ')
else:
    rows = []
    TG = [g for g in ['test รวม', 'test แปลงเดิม', 'test แปลงใหม่', 'test เดิม (weed-only)'] if g in SETS]
    for k, (pt, s_) in CONFIGS.items():
        r = {'config': k, 'WINNER': '✅' if k == WINNER else '', 'val mAP50': v[k]}
        PRED[(k, 'test รวม')], _ = predict(pt, SETS['test รวม'], s_)
        for g in TG:
            if g == 'test เดิม (weed-only)': PRED[(k, g)], _ = predict(pt, SETS[g], s_)
            elif g != 'test รวม': PRED[(k, g)] = {p: PRED[(k, 'test รวม')][p] for p in SETS[g]}
            sc = score(PRED[(k, g)])
            r[f'{g} mAP50'] = sc['mAP50']
            if g in ('test รวม', 'test แปลงใหม่'): r.update({f'{g} mAP50-95': sc['mAP50_95'], f'{g} P': sc['P'], f'{g} R': sc['R']})
        r['วินาที/ภาพ (T4)'] = round(SEC[k], 2); rows.append(r); print('✅', k, flush=True)
    TEST = pd.DataFrame(rows); display(TEST); TEST.to_csv(f'{W}/v7_test.csv', index=False)
    print({g: len(SETS[g]) for g in TG})

In [ ]:
# 8) ระดับช่องกริด 12 คอลัมน์ · conf 0.25 (นิยามเดียวกับ grid_eval) — ไม่สนว่าตีกรอบแบบกอหรือต้น → ตัวเทียบที่ยุติธรรมที่สุด
if not TRAIN_DONE:
    print('ข้าม — ยังเทรนไม่ครบ')
else:
    def grid_mask(boxes, W_, H_, cols=12, conf_th=0.0):
        cell = W_ / cols; nr = max(1, round(H_ / cell)); ch = H_ / nr
        m = np.zeros((nr, cols), bool)
        for x1, y1, x2, y2, c in np.asarray(boxes).reshape(-1, 5):
            if c < conf_th: continue
            m[int(max(y1, 0)//ch):int(min(y2, H_-1)//ch)+1, int(max(x1, 0)//cell):int(min(x2, W_-1)//cell)+1] = True
        return m
    rows = []
    for k, g_ in [(k, g_) for k in CONFIGS if 'SAHI 1024' in k for g_ in ['test รวม', 'test แปลงเดิม', 'test แปลงใหม่'] if g_ in SETS]:
        TP = FP = FN = TN = 0
        for p, (d, W_, H_) in PRED[(k, g_)].items():
            g = gt_boxes(p, W_, H_)
            gm = grid_mask(np.c_[g, np.ones(len(g))], W_, H_); dm = grid_mask(d, W_, H_, 12, 0.25)
            TP += int((gm & dm).sum()); FP += int((~gm & dm).sum()); FN += int((gm & ~dm).sum()); TN += int((~gm & ~dm).sum())
        P = TP/max(TP+FP, 1); R = TP/max(TP+FN, 1)
        rows.append({'config': k, 'ชุด': g_, 'Precision': round(P*100, 1), 'Recall': round(R*100, 1), 'F1': round(2*P*R/max(P+R, 1e-9)*100, 1),
                     'ลดพื้นที่พ่น (%)': round((1-(TP+FP)/(TP+FP+FN+TN))*100, 1)})
    GRID = pd.DataFrame(rows); display(GRID); GRID.to_csv(f'{W}/v7_grid.csv', index=False)

In [ ]:
# 9) รวมไฟล์ผลไว้ดาวน์โหลด (แท็บ Output) — โมเดล v7_best_sofar.pt อยู่แยกข้างนอก
if not TRAIN_DONE:
    print('ข้าม — ยังเทรนไม่ครบ')
else:
    R = f'{W}/v7_results'; os.makedirs(R, exist_ok=True)
    for f in ['v7_val.csv', 'v7_test.csv', 'v7_grid.csv']: shutil.copy(f'{W}/{f}', R)
    shutil.copy(f'{W}/v7_results_progress.csv', f'{R}/train_results_last_round.csv')
    if os.path.exists(f'{W}/v7_results_previous_rounds.csv'): shutil.copy(f'{W}/v7_results_previous_rounds.csv', R)
    open(f'{R}/roboflow_version.txt', 'w').write(str(RF_VER))
    shutil.make_archive(f'{W}/v7_results', 'zip', R)
    print('ส่ง v7_results.zip มาได้เลย')